<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/Models_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU, Conv1D, MaxPooling1D, Flatten, Bidirectional, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# --- THE FAIRNESS LOCK ---
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def prep_data(df, feature_cols, target_col='Close', window_size=10, test_size=0.2):
    # 1. Isolate the data
    data = df[feature_cols].values
    target = df[[target_col]].values

    # 2. Split indices
    split_idx = int(len(data) * (1 - test_size))

    # 3. Scale Features (Fit ONLY on training data to prevent leakage)
    scaler_X = MinMaxScaler()
    train_X_scaled = scaler_X.fit_transform(data[:split_idx])
    test_X_scaled = scaler_X.transform(data[split_idx:])
    scaled_data = np.vstack((train_X_scaled, test_X_scaled))

    # 4. Scale Target (Fit ONLY on training target)
    scaler_y = MinMaxScaler()
    train_y_scaled = scaler_y.fit_transform(target[:split_idx])
    test_y_scaled = scaler_y.transform(target[split_idx:])
    scaled_target = np.vstack((train_y_scaled, test_y_scaled))

    # 5. Create 10-day Sequences
    X, y = [], []
    for i in range(window_size, len(scaled_data)):
        X.append(scaled_data[i-window_size:i])
        y.append(scaled_target[i])

    X, y = np.array(X), np.array(y)

    # 6. Final Split
    new_split_idx = split_idx - window_size
    X_train, y_train = X[:new_split_idx], y[:new_split_idx]
    X_test, y_test = X[new_split_idx:], y[new_split_idx:]

    # Also save the true unscaled test values for charting later
    y_test_true = target[split_idx:]

    return X_train, y_train, X_test, y_test, scaler_y, y_test_true

print("Data Pipeline Engine Ready.")

In [ ]:
from tensorflow.keras.layers import Input, Dropout

def build_model(model_type, input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))

    if model_type == 'RNN':
        model.add(SimpleRNN(64, return_sequences=True))
        model.add(Dropout(0.2))
        model.add(SimpleRNN(32))

    elif model_type == 'LSTM':
        model.add(LSTM(64, return_sequences=True))
        model.add(Dropout(0.2))
        model.add(LSTM(32))

    elif model_type == 'GRU':
        model.add(GRU(64, return_sequences=True))
        model.add(Dropout(0.2))
        model.add(GRU(32))

    elif model_type == 'CNN-LSTM':
        model.add(Conv1D(filters=64, kernel_size=2, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(LSTM(32))

    elif model_type == 'BiLSTM':
        model.add(Bidirectional(LSTM(units=64, return_sequences=True)))
        model.add(Dropout(0.2))
        model.add(LSTM(units=32, return_sequences=False))
        model.add(Dropout(0.2))

    # Standard output layers for all models
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mean_squared_error')
    return model


In [ ]:

df = pd.read_csv('BBCA_Dataset.csv')

features_normal = ['Open', 'High', 'Low', 'Close', 'Volume']
features_assisted = ['Open', 'High', 'Low', 'Close', 'Volume', 'Sentiment_Score']

models_to_test = ['RNN', 'LSTM', 'GRU', 'CNN-LSTM', 'BiLSTM']
results = []
predictions_dict = {}

# We use EarlyStopping to prevent overfitting and ensure fairness in training time
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

for model_name in models_to_test:
    print(f"\n{'='*60}\n STARTING BENCHMARK: {model_name}\n{'='*60}")
    predictions_dict[model_name] = {}

    for mode, features in zip(['Normal', 'Assisted'], [features_normal, features_assisted]):
        print(f"--> Training {mode} Mode ({len(features)} features)...")

        # 1. Prep Data
        X_train, y_train, X_test, y_test, scaler_y, y_test_true = prep_data(df, features)

        # 2. Build Model
        model = build_model(model_name, (X_train.shape[1], X_train.shape[2]))

        # 3. Train
        model.fit(X_train, y_train, epochs=100, batch_size=16,
                  validation_data=(X_test, y_test),
                  callbacks=[early_stop], verbose=0)

        # 4. Predict & Inverse Scale to get real Rupiah values
        pred_scaled = model.predict(X_test, verbose=0)
        pred_true = scaler_y.inverse_transform(pred_scaled)

        # --- 5. MULTI-METRIC SCORING ---
        y_actual = y_test_true.flatten()
        y_pred = pred_true.flatten()

        # A. RMSE (Root Mean Squared Error)
        rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

        # B. MAE (Mean Absolute Error - Average error in Rupiah)
        mae = mean_absolute_error(y_actual, y_pred)

        # C. MAPE (Mean Absolute Percentage Error)
        mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100

        # D. R2 Score (Coefficient of Determination)
        r2 = r2_score(y_actual, y_pred)

        # E. Directional Accuracy (Win Rate)
        # We check if the predicted movement (Up/Down) matches the actual movement
        actual_diff = np.diff(y_actual)
        # Predicted diff: predicted price today vs actual price yesterday
        pred_diff = y_pred[1:] - y_actual[:-1]
        dir_accuracy = np.mean((actual_diff > 0) == (pred_diff > 0)) * 100

        print(f" Result: RMSE: {rmse:.2f} | DA (Win Rate): {dir_accuracy:.2f}%")

        # Store all results for the final table
        results.append({
            'Model': model_name,
            'Mode': mode,
            'RMSE': round(rmse, 2),
            'MAE': round(mae, 2),
            'MAPE (%)': round(mape, 2),
            'R2 Score': round(r2, 4),
            'Win Rate (%)': round(dir_accuracy, 2)
        })
        predictions_dict[model_name][mode] = y_pred

# Save actuals for plotting later
actual_prices = y_test_true.flatten()

# Create a clean results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("FINAL BENCHMARKING REPORT: ALL MODELS EVALUATED")
print("="*60)

# Display the main comparison table
# We pivot this table to make it very easy to compare metrics side-by-side
comparison_table = results_df.sort_values(by=['Model', 'Mode'])
display(comparison_table)

comparison_table.to_csv('BBCA_Model_Comparison_Results.csv', index=False)

In [ ]:
time_steps = range(len(actual_prices))

fig, axes = plt.subplots(5, 1, figsize=(14, 25))
fig.tight_layout(pad=6.0)

for idx, model_name in enumerate(models_to_test):
    ax = axes[idx]

    # Plot Actual
    ax.plot(time_steps, actual_prices, label='Actual Price', color='black', linewidth=2, linestyle='dashed')

    # Plot Normal
    ax.plot(time_steps, predictions_dict[model_name]['Normal'], label=f'{model_name} (Normal)', color='red', alpha=0.7)

    # Plot Assisted
    ax.plot(time_steps, predictions_dict[model_name]['Assisted'], label=f'{model_name} (Sentiment Assisted)', color='green', linewidth=2)

    ax.set_title(f"{model_name}: Normal vs Sentiment Assisted", fontsize=16, fontweight='bold')
    ax.set_ylabel("Price (IDR)", fontsize=12)
    ax.legend(loc='upper right')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# --- PLOT 1: RMSE  ---
sns.barplot(data=results_df, x='Model', y='RMSE', hue='Mode', palette=['#E74C3C', '#2ECC71'], ax=ax1)
ax1.set_title("RMSE Comparison (Error Magnitude)", fontsize=14, fontweight='bold')
ax1.set_ylabel("RMSE (Rupiah)")

# Add labels to RMSE bars
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.0f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontsize=10)

# --- PLOT 2: WIN RATE ---
sns.barplot(data=results_df, x='Model', y='Win Rate (%)', hue='Mode', palette=['#34495E', '#3498DB'], ax=ax2)
ax2.set_title("Directional Accuracy (Win Rate %)", fontsize=14, fontweight='bold')
ax2.set_ylabel("Accuracy %")
ax2.set_ylim(0, 100)

# Add labels to Win Rate bars
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()